In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from spreg import GM_Error_Het, GM_Error_Hom, ML_Error
from esda.moran import Moran
import libpysal
from libpysal.weights import Queen
import matplotlib.pyplot as plt
import geopandas as gpd
import scipy as sp
import arviz as az

RANDOM_SEED = 5781

# Transportation

## Data

In [2]:
tran_par_file = '/work/hawkins_lab/vulcan/data/vulcan/parquet/vulcan_ONR_epa_climate.parquet'
tran_df = gpd.read_parquet(tran_par_file)
tran_wgt_file = '/work/hawkins_lab/vulcan/results/tran_weights.csv'
tran_wgt_df = pd.read_csv(tran_wgt_file)
# Create columns that assign decile numbers for each treatment variable
tran_df['treat_density'] = tran_df['d1a'] /1000
tran_df['treat_diversity'] = tran_df['d2b_e5mixa']
tran_df['treat_design'] = tran_df['d3a']
tran_df['treat_distance'] = tran_df['d4a'].replace(-99999,1500)  /1000
tran_df['treat_destination'] = tran_df['d5ar']  /1000

tran_df = pd.concat([tran_df,tran_wgt_df], axis=1)
tran_df = tran_df[(tran_df["value_weig"]>0) & (tran_df["totpop"]>0)]
tran_df["d1aa_cbsa"] = tran_df["totpop_cbsa"]/tran_df["ac_unpr_cbsa"] # true cbsa density

In [3]:
tran_df.groupby('cbsa').size()

cbsa
1.11          7
1.13         19
1.19         19
1.23         15
1.25         24
           ... 
49660.00    520
49700.00    111
49740.00    140
49780.00     75
49820.00      9
Length: 2013, dtype: int64

In [4]:
for c in ["treat_density","treat_diversity","treat_design","treat_distance","treat_destination", 
          "totpop_cbsa", "d1aa_cbsa","d2b_e5mix_cbsa","d3a_cbsa", "d4a_cbsa", "d5ar_cbsa"]:
    mu, sd = tran_df[c].mean(), tran_df[c].std()
    tran_df[c+"_z"] = (tran_df[c] - mu) / (sd if sd!=0 else 1.0)

for g in ["dens","div","des","dist","dest"]:
    mu, sd = tran_df[g].mean(), tran_df[g].std()
    tran_df["gps_"+g+"_z"] = (tran_df[g] - mu) / (sd if sd!=0 else 1.0)

tran_df["treat_density_ps"]   = tran_df["treat_density_z"]   * tran_df["gps_dens_z"]
tran_df["treat_diversity_ps"]  = tran_df["treat_diversity_z"] * tran_df["gps_div_z"]
tran_df["treat_design_ps"]     = tran_df["treat_design_z"]    * tran_df["gps_des_z"]
tran_df["treat_distance_ps"]  = tran_df["treat_distance_z"]  * tran_df["gps_dist_z"]
tran_df["treat_destination_ps"]= tran_df["treat_destination_z"] * tran_df["gps_dest_z"]

predictors = ["totpop_cbsa_z",
              "d1aa_cbsa_z",
              # "d1a_cbsa",
              "d2b_e5mix_cbsa_z",
              "d3a_cbsa_z",
              "d4a_cbsa_z", 
              "d5ar_cbsa_z",
              # "vmt_per_wo", # Add this one as the last of 3 transport-specific control variables from EPA data # Statistically insign
              "gps_dens_z",
              "gps_div_z",
              "gps_des_z",
              "gps_dist_z",
              "gps_dest_z",
              "treat_density_z",
              "treat_diversity_z",
              "treat_design_z",
              "treat_distance_z",
              "treat_destination_z",
              "treat_density_ps",
              "treat_diversity_ps",
              "treat_design_ps",
              "treat_distance_ps",
              "treat_destination_ps"]

X = tran_df.copy()

y = np.log(tran_df["value_weig"].replace(0, 10**-6) / tran_df["totpop"].replace(0, np.nan))

# X.loc[:,log_predictors] = np.log(X.loc[:,log_predictors].replace(0, 10**-6) ) # replace 0 with 10**-6

X1 = X.loc[:, predictors]
X_fe = pd.get_dummies(tran_df["cbsa_eig"], prefix="metro", drop_first=True).astype(int)
# Combine with covariates
X1_fe = pd.concat([X1, X_fe], axis=1)


# Add SEM term

In [5]:
# model = GM_Error_Het(y, X_slx, w=W, name_y='y', name_x=list(X_slx.columns))
# print(model.summary)

In [6]:
# residuals = model.y - model.predy

# mi = Moran(residuals, W)
# print(mi.I, mi.p_sim)

In [7]:
W = Queen.from_dataframe(tran_df)
W.transform = 'r'

WX = pd.DataFrame(
    {f"W_{col}": W.sparse.dot(X1_fe[col]) for col in ["treat_density_z","treat_diversity_z","treat_design_z","treat_distance_z","treat_destination_z"]},
    index=X1_fe.index
)

X_slx = pd.concat([X1_fe, WX], axis=1)

/tmp/ipykernel_3864466/53609628.py:1: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  W = Queen.from_dataframe(tran_df)
/home/jfhawkin/software/miniforge3/envs/pymc_env/lib/python3.13/site-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 27 disconnected components.
 There are 12 islands with ids: 18616, 29427, 32710, 84610, 89731, 94628, 126088, 134362, 137261, 150235, 172351, 207415.
  W.__init__(self, neighbors, ids=ids, **kw)


('WARNING: ', 18616, ' is an island (no neighbors)')
('WARNING: ', 29427, ' is an island (no neighbors)')
('WARNING: ', 32710, ' is an island (no neighbors)')
('WARNING: ', 84610, ' is an island (no neighbors)')
('WARNING: ', 89731, ' is an island (no neighbors)')
('WARNING: ', 94628, ' is an island (no neighbors)')
('WARNING: ', 126088, ' is an island (no neighbors)')
('WARNING: ', 134362, ' is an island (no neighbors)')
('WARNING: ', 137261, ' is an island (no neighbors)')
('WARNING: ', 150235, ' is an island (no neighbors)')
('WARNING: ', 172351, ' is an island (no neighbors)')
('WARNING: ', 207415, ' is an island (no neighbors)')


In [8]:
model = GM_Error_Het(y, X_slx, w=W, name_y='y', name_x=list(X_slx.columns))
print(model.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: GM SPATIALLY WEIGHTED LEAST SQUARES (HET)
------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :           y                Number of Observations:      215309
Mean dependent var  :      0.9804                Number of Variables   :         361
S.D. dependent var  :      1.1514                Degrees of Freedom    :      214948
Pseudo R-squared    :      0.2670
N. of iterations    :           1                Step1c computed       :          No

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------------------------------------------------------
            CONSTANT         1.33377         0.05703        23.38542         0.00000
       totpop_cbsa_z        -0.02989         0.06042    

In [9]:
# model = GM_Error(y, X_slx, w=W, name_y='y', name_x=list(X_slx.columns), method='LU')
# print(model.summary)

In [10]:
residuals = model.y - model.predy

mi = Moran(residuals, W)
print(mi.I, mi.p_sim)

0.3839296951790668 0.001


In [ ]:
# # 2. Extract needed pieces
# u = model.u  # residuals
# params = model.betas.flatten()

# # Build design matrix (PySAL already includes constant if you passed one)
# X = model.x

# # 3. Compute (X'X)^(-1)
# XtX_inv = np.linalg.inv(X.T @ X)

# # 4. Compute cluster meat
# clusters = pd.Series(clusters)  # ensure pandas for grouping
# meat = np.zeros((X.shape[1], X.shape[1]))

# for g in clusters.unique():
#     idx = clusters == g
#     Xg = X[idx]
#     ug = np.atleast_1d(u[idx])
#     meat += Xg.T @ (ug[:, None] @ ug[None, :]) @ Xg

# # 5. Sandwich covariance
# V = XtX_inv @ meat @ XtX_inv

# # 6. Clustered standard errors
# se_clustered = np.sqrt(np.diag(V))

# print(se_clustered)

# SEM Distance Band

In [ ]:
from libpysal.weights import DistanceBand

W1 = DistanceBand.from_dataframe(
    tran_df,
    threshold=10000,
    binary=False,
    alpha=-1.0   # exponential decay
)
W1.transform = 'r'

WX = pd.DataFrame(
    {f"W_{col}": W1.sparse.dot(X1_fe[col]) for col in ["treat_density_z","treat_diversity_z","treat_design_z","treat_distance_z","treat_destination_z"]},
    index=X1_fe.index
)

X_slx = pd.concat([X1_fe, WX], axis=1)

/home/jfhawkin/software/miniforge3/envs/pymc_env/lib/python3.13/site-packages/scipy/sparse/_data.py:128: RuntimeWarning: divide by zero encountered in reciprocal
  return self._with_data(data ** n)
/home/jfhawkin/software/miniforge3/envs/pymc_env/lib/python3.13/site-packages/libpysal/weights/util.py:826: UserWarning: The weights matrix is not fully connected: 
 There are 7581 disconnected components.
 There are 5553 islands with ids: 27, 29, 30, 31, 32, 33, 34, 40, 109, 126, 129, 130, 131, 132, 133, 134, 156, 157, 160, 199, 202, 203, 204, 205, 211, 224, 317, 328, 390, 391, 392, 400, 401, 408, 409, 411, 412, 413, 415, 421, 422, 425, 426, 429, 436, 437, 449, 450, 465, 476, 539, 540, 541, 543, 546, 550, 556, 559, 562, 569, 605, 607, 608, 609, 610, 743, 745, 854, 855, 867, 869, 983, 1035, 1036, 1037, 1045, 1046, 1056, 1057, 1058, 1072, 1155, 1158, 1160, 1762, 1946, 1949, 1953, 1954, 1958, 1975, 1976, 1977, 2176, 2177, 2180, 2181, 2182, 2464, 2555, 2556, 2557, 2559, 2575, 2756, 2757, 2850, 

('WARNING: ', 27, ' is an island (no neighbors)')
('WARNING: ', 29, ' is an island (no neighbors)')
('WARNING: ', 30, ' is an island (no neighbors)')
('WARNING: ', 31, ' is an island (no neighbors)')
('WARNING: ', 32, ' is an island (no neighbors)')
('WARNING: ', 33, ' is an island (no neighbors)')
('WARNING: ', 34, ' is an island (no neighbors)')
('WARNING: ', 40, ' is an island (no neighbors)')
('WARNING: ', 109, ' is an island (no neighbors)')
('WARNING: ', 126, ' is an island (no neighbors)')
('WARNING: ', 129, ' is an island (no neighbors)')
('WARNING: ', 130, ' is an island (no neighbors)')
('WARNING: ', 131, ' is an island (no neighbors)')
('WARNING: ', 132, ' is an island (no neighbors)')
('WARNING: ', 133, ' is an island (no neighbors)')
('WARNING: ', 134, ' is an island (no neighbors)')
('WARNING: ', 156, ' is an island (no neighbors)')
('WARNING: ', 157, ' is an island (no neighbors)')
('WARNING: ', 160, ' is an island (no neighbors)')
('WARNING: ', 199, ' is an island (no n

In [ ]:
# model = GM_Error_Het(y, X_slx, w=W1, name_y='y', name_x=list(X_slx.columns))
# print(model.summary)

In [ ]:
# model = GM_Error(y, X_slx, w=W1, name_y='y', name_x=list(X_slx.columns))
# print(model.summary)

In [ ]:
# model = GM_Error_Hom(y, X_slx, w=W1, name_y='y', name_x=list(X_slx.columns))
# print(model.summary)

In [ ]:
model = GM_Error_Het(y, X_slx, w=W1, name_y='y', name_x=list(X_slx.columns))
print(model.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: GM SPATIALLY WEIGHTED LEAST SQUARES (HET)
------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :           y                Number of Observations:      215309
Mean dependent var  :      0.9804                Number of Variables   :         361
S.D. dependent var  :      1.1514                Degrees of Freedom    :      214948
Pseudo R-squared    :      0.2621
N. of iterations    :           1                Step1c computed       :          No

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------------------------------------------------------
            CONSTANT         1.57904         0.05455        28.94543         0.00000
       totpop_cbsa_z        -0.04383         0.06543    

In [ ]:
residuals = model.y - model.predy

mi = Moran(residuals, W1)
print(mi.I, mi.p_sim)

0.17036789603689576 0.001
